In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto, kstest


NONCRASH_CSV = "merged_filtered_data(april_may_june).csv"
TTC_COL = "TTC"
LTTB_COL = "LTTB"
THRESHOLD_Q = 5

OUT_DIR = "evt_outputs_noncrash"


def clean_positive(series):
    x = pd.to_numeric(series, errors="coerce").values
    x = x[np.isfinite(x)]
    return x[x > 0]

def lower_tail_exceedances(x, u):
    """
    Lower-tail POT exceedances (danger = small X):
      Y = u - X  |  X < u
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    exc = (u - x[x < u])
    return exc[exc > 0]

def fit_gpd(exceed):
    """
    Fit GPD to exceedances with loc fixed at 0:
      Y ~ GPD(xi, sigma), loc=0
    """
    if len(exceed) < 5:
        raise ValueError(f"Too few exceedances to fit GPD: n={len(exceed)}")
    xi, loc, sigma = genpareto.fit(exceed, floc=0.0)
    return xi, loc, sigma

def plot_emp_vs_gpd_cdf(exceed, params, title, xlabel, filename):
    xi, loc, sigma = params
    xs = np.sort(exceed)
    ecdf = np.arange(1, len(xs) + 1) / len(xs)

    xgrid = np.linspace(0, xs.max(), 400) if xs.max() > 0 else np.linspace(0, 1, 400)
    tcdf = genpareto.cdf(xgrid, c=xi, loc=loc, scale=sigma)

    plt.figure()
    plt.plot(xs, ecdf, label="Empirical CDF")
    plt.plot(xgrid, tcdf, label="GPD CDF (fit)")
    plt.xlabel(xlabel)
    plt.ylabel("CDF")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()

def plot_qq(exceed, params, title, filename):
    xi, loc, sigma = params
    xs = np.sort(exceed)
    n = len(xs)
    p = (np.arange(1, n + 1) - 0.5) / n
    theo = genpareto.ppf(p, c=xi, loc=loc, scale=sigma)

    plt.figure()
    plt.plot(theo, xs, marker="o", linestyle="None", label="QQ points")
    mn = min(theo.min(), xs.min())
    mx = max(theo.max(), xs.max())
    plt.plot([mn, mx], [mn, mx], label="Identity line")
    plt.xlabel("Theoretical quantiles (GPD)")
    plt.ylabel("Empirical quantiles")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()

def ks_test(exceed, params):
    xi, loc, sigma = params
    D, p = kstest(exceed, lambda t: genpareto.cdf(t, c=xi, loc=loc, scale=sigma))
    return D, p

# ---------- Setup output dir ----------
os.makedirs(OUT_DIR, exist_ok=True)
print("Working directory:", os.getcwd())
print("Figures will be saved to:", os.path.abspath(OUT_DIR))

# ---------- Load NON-CRASH ----------
df_nc = pd.read_csv(NONCRASH_CSV)
ttc_nc = clean_positive(df_nc[TTC_COL])
lttb_nc = clean_positive(df_nc[LTTB_COL])

# ---------- Thresholds ----------
u_ttc = np.percentile(ttc_nc, THRESHOLD_Q)
u_lttb = np.percentile(lttb_nc, THRESHOLD_Q)

# ---------- Exceedances ----------
exc_ttc_nc = lower_tail_exceedances(ttc_nc, u_ttc)
exc_lttb_nc = lower_tail_exceedances(lttb_nc, u_lttb)

# ---------- Fit ----------
params_ttc_nc = fit_gpd(exc_ttc_nc)
params_lttb_nc = fit_gpd(exc_lttb_nc)

print("=== NON-CRASH thresholds ===")
print("u_TTC  =", u_ttc,  "| N_exceed =", len(exc_ttc_nc))
print("u_LTTB =", u_lttb, "| N_exceed =", len(exc_lttb_nc))

print("\n=== NON-CRASH GPD params (xi, loc, sigma) ===")
print("TTC :", params_ttc_nc)
print("LTTB:", params_lttb_nc)


# TTC
plot_emp_vs_gpd_cdf(
    exc_ttc_nc, params_ttc_nc,
    title=f"Non-crash TTC exceedances — Empirical vs GPD CDF (n={len(exc_ttc_nc)})",
    xlabel="TTC exceedance (u - TTC)",
    filename=os.path.join(OUT_DIR, "TTC_nc_emp_vs_gpd_cdf.png"),
)
plot_qq(
    exc_ttc_nc, params_ttc_nc,
    title="Non-crash TTC exceedances — QQ plot (GPD)",
    filename=os.path.join(OUT_DIR, "TTC_nc_qq.png"),
)
D_ttc, p_ttc = ks_test(exc_ttc_nc, params_ttc_nc)
print("KS test (non-crash TTC): D =", D_ttc, ", p =", p_ttc)

# LTTB
plot_emp_vs_gpd_cdf(
    exc_lttb_nc, params_lttb_nc,
    title=f"Non-crash LTTB exceedances — Empirical vs GPD CDF (n={len(exc_lttb_nc)})",
    xlabel="LTTB exceedance (u - LTTB)",
    filename=os.path.join(OUT_DIR, "LTTB_nc_emp_vs_gpd_cdf.png"),
)
plot_qq(
    exc_lttb_nc, params_lttb_nc,
    title="Non-crash LTTB exceedances — QQ plot (GPD)",
    filename=os.path.join(OUT_DIR, "LTTB_nc_qq.png"),
)
D_lttb, p_lttb = ks_test(exc_lttb_nc, params_lttb_nc)
print("KS test (non-crash LTTB): D =", D_lttb, ", p =", p_lttb)


print("\nCOPY THESE FOR CRASH MODELING:")
print("u_ttc =", u_ttc)
print("u_lttb =", u_lttb)


Working directory: /content
Figures will be saved to: /content/evt_outputs_noncrash
=== NON-CRASH thresholds ===
u_TTC  = 1.125873866 | N_exceed = 97
u_LTTB = 2.040754798 | N_exceed = 97

=== NON-CRASH GPD params (xi, loc, sigma) ===
TTC : (np.float64(-1.2209480742758783), 0.0, np.float64(1.3667576320843915))
LTTB: (np.float64(-0.8638282350565922), 0.0, np.float64(1.1700685974751988))
KS test (non-crash TTC): D = 0.08437177683090763 , p = 0.4693573703357864
KS test (non-crash LTTB): D = 0.10907718425524282 , p = 0.1846139246719185

COPY THESE FOR CRASH MODELING:
u_ttc = 1.125873866
u_lttb = 2.040754798


In [ ]:

# Use SAME thresholds from NON-CRASH for comparability


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto, kstest


CRASH_CSV = "/content/TTC_results.csv"
TTC_COL = "TTC"
LTTB_COL = "LTTB"


u_ttc = 1.125873865737416
u_lttb = 2.0407547984513457


OUT_DIR = "evt_outputs_crash"


def clean_positive(series):
    x = pd.to_numeric(series, errors="coerce").values
    x = x[np.isfinite(x)]
    return x[x > 0]

def lower_tail_exceedances(x, u):
    """
    Lower-tail POT exceedances (danger = small X):
      Y = u - X  |  X < u
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    exc = (u - x[x < u])
    return exc[exc > 0]

def fit_gpd(exceed):
    """
    Fit GPD to exceedances with loc fixed at 0:
      Y ~ GPD(xi, sigma), loc=0
    """
    if len(exceed) < 5:
        raise ValueError(f"Too few exceedances to fit GPD: n={len(exceed)}")
    xi, loc, sigma = genpareto.fit(exceed, floc=0.0)
    return xi, loc, sigma

def plot_emp_vs_gpd_cdf(exceed, params, title, xlabel, filename):
    xi, loc, sigma = params
    xs = np.sort(exceed)
    ecdf = np.arange(1, len(xs) + 1) / len(xs)

    xgrid = np.linspace(0, xs.max(), 400) if xs.max() > 0 else np.linspace(0, 1, 400)
    tcdf = genpareto.cdf(xgrid, c=xi, loc=loc, scale=sigma)

    plt.figure()
    plt.plot(xs, ecdf, label="Empirical CDF")
    plt.plot(xgrid, tcdf, label="GPD CDF (fit)")
    plt.xlabel(xlabel)
    plt.ylabel("CDF")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()

def plot_qq(exceed, params, title, filename):
    xi, loc, sigma = params
    xs = np.sort(exceed)
    n = len(xs)
    p = (np.arange(1, n + 1) - 0.5) / n
    theo = genpareto.ppf(p, c=xi, loc=loc, scale=sigma)

    plt.figure()
    plt.plot(theo, xs, marker="o", linestyle="None", label="QQ points")
    mn = min(theo.min(), xs.min())
    mx = max(theo.max(), xs.max())
    plt.plot([mn, mx], [mn, mx], label="Identity line")
    plt.xlabel("Theoretical quantiles (GPD)")
    plt.ylabel("Empirical quantiles")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()

def ks_test(exceed, params):
    xi, loc, sigma = params
    D, p = kstest(exceed, lambda t: genpareto.cdf(t, c=xi, loc=loc, scale=sigma))
    return D, p


os.makedirs(OUT_DIR, exist_ok=True)
print("Working directory:", os.getcwd())
print("Figures will be saved to:", os.path.abspath(OUT_DIR))

df_c = pd.read_csv(CRASH_CSV)
ttc_c = clean_positive(df_c[TTC_COL])
lttb_c = clean_positive(df_c[LTTB_COL])

exc_ttc_c = lower_tail_exceedances(ttc_c, u_ttc)
exc_lttb_c = lower_tail_exceedances(lttb_c, u_lttb)


params_ttc_c = fit_gpd(exc_ttc_c)
params_lttb_c = fit_gpd(exc_lttb_c)

print("=== CRASH thresholds (reused) ===")
print("u_TTC  =", u_ttc,  "| N_exceed =", len(exc_ttc_c))
print("u_LTTB =", u_lttb, "| N_exceed =", len(exc_lttb_c))

print("\n=== CRASH GPD params (xi, loc, sigma) ===")
print("TTC :", params_ttc_c)
print("LTTB:", params_lttb_c)


# TTC
plot_emp_vs_gpd_cdf(
    exc_ttc_c, params_ttc_c,
    title=f"Crash TTC exceedances — Empirical vs GPD CDF (n={len(exc_ttc_c)})",
    xlabel="TTC exceedance (u - TTC)",
    filename=os.path.join(OUT_DIR, "TTC_crash_emp_vs_gpd_cdf.png"),
)
plot_qq(
    exc_ttc_c, params_ttc_c,
    title="Crash TTC exceedances — QQ plot (GPD)",
    filename=os.path.join(OUT_DIR, "TTC_crash_qq.png"),
)
D_ttc, p_ttc = ks_test(exc_ttc_c, params_ttc_c)
print("KS test (crash TTC): D =", D_ttc, ", p =", p_ttc)

# LTTB
plot_emp_vs_gpd_cdf(
    exc_lttb_c, params_lttb_c,
    title=f"Crash LTTB exceedances — Empirical vs GPD CDF (n={len(exc_lttb_c)})",
    xlabel="LTTB exceedance (u - LTTB)",
    filename=os.path.join(OUT_DIR, "LTTB_crash_emp_vs_gpd_cdf.png"),
)
plot_qq(
    exc_lttb_c, params_lttb_c,
    title="Crash LTTB exceedances — QQ plot (GPD)",
    filename=os.path.join(OUT_DIR, "LTTB_crash_qq.png"),
)
D_lttb, p_lttb = ks_test(exc_lttb_c, params_lttb_c)
print("KS test (crash LTTB): D =", D_lttb, ", p =", p_lttb)


Working directory: /content
Figures will be saved to: /content/evt_outputs_crash
=== CRASH thresholds (reused) ===
u_TTC  = 1.125873865737416 | N_exceed = 60
u_LTTB = 2.0407547984513457 | N_exceed = 65

=== CRASH GPD params (xi, loc, sigma) ===
TTC : (np.float64(-0.9305254945478006), 0.0, np.float64(1.0337204744509063))
LTTB: (np.float64(-1.6356895335529635), 0.0, np.float64(3.3133968748574256))
KS test (crash TTC): D = 0.05165088662939374 , p = 0.9946861215612153
KS test (crash LTTB): D = 0.19499991405593137 , p = 0.012194170776612667


In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


NONCRASH_CSV = "merged_filtered_data(april_may_june).csv"
CRASH_CSV = "/content/TTC_results.csv"

TTC_COL = "TTC"
LTTB_COL = "LTTB"

THRESHOLD_Q = 5
XLIM = (0, 10)
BINS = 60


OUT_TTC_FIG = "Fig_TTC_EVT_density.png"
OUT_LTTB_FIG = "Fig_LTTB_EVT_density.png"
DPI = 300


def clean_positive(series):
    x = pd.to_numeric(series, errors="coerce").values
    x = x[np.isfinite(x)]
    return x[x > 0]

def lower_tail_exceedances(x, u):
    """
    Lower-tail POT:
    For X<u, exceedance Y = u - X  (so Y>0)
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    exc = (u - x[x < u])
    return exc[exc > 0]

def fit_gpd(exceed):
    """
    Fit GPD to exceedances with loc fixed at 0
    """
    xi, loc, sigma = genpareto.fit(exceed, floc=0.0)
    return xi, loc, sigma

def tail_pdf_on_invx_grid(z, u, gpd_params):
    """
    Lower-tail POT: Y = u - X | X<u  ~ GPD
    For Z = 1/X:
      f_Z(z) = f_X(1/z) * (1/z^2)

    Tail domain X<=u -> z >= 1/u
    """
    xi, loc, sigma = gpd_params
    z = np.asarray(z, dtype=float)

    x = 1.0 / np.clip(z, 1e-12, None)
    y = u - x

    fY = genpareto.pdf(y, c=xi, loc=loc, scale=sigma)
    fZ = fY * (1.0 / (z**2))

    fZ = np.where(z >= (1.0 / u), fZ, np.nan)
    return fZ

def chen_style_density_plot(
    z_nc, z_c, u, params_nc, params_c,
    title, xlabel, xlim=(0, 10),
    inset1_xlim=(0, 2.5), inset2_xlim=(6, 10),
    bins=60,
    savepath=None, dpi=300
):
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.hist(z_nc, bins=bins, range=xlim, density=True, alpha=0.35, label="Non-crash condition")
    ax.hist(z_c,  bins=bins, range=xlim, density=True, alpha=0.35, label="Crash condition")

    z_grid = np.linspace(xlim[0] + 1e-6, xlim[1], 800)
    ax.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_nc), label="Non-crash (EVT fit)")
    ax.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_c),  label="Crash (EVT fit)")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Probability density")
    ax.set_xlim(xlim)
    ax.set_title(title)
    ax.legend(loc="upper right")

    axins1 = inset_axes(ax, width="30%", height="30%", loc="upper left", borderpad=1.2)
    axins1.hist(z_nc, bins=bins, range=xlim, density=True, alpha=0.35)
    axins1.hist(z_c,  bins=bins, range=xlim, density=True, alpha=0.35)
    axins1.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_nc))
    axins1.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_c))
    axins1.set_xlim(inset1_xlim)
    axins1.set_xticks([])
    axins1.set_yticks([])
    mark_inset(ax, axins1, loc1=2, loc2=4, fc="none", ec="0.5")

    axins2 = inset_axes(ax, width="30%", height="30%", loc="center right", borderpad=1.2)
    axins2.hist(z_nc, bins=bins, range=xlim, density=True, alpha=0.35)
    axins2.hist(z_c,  bins=bins, range=xlim, density=True, alpha=0.35)
    axins2.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_nc))
    axins2.plot(z_grid, tail_pdf_on_invx_grid(z_grid, u, params_c))
    axins2.set_xlim(inset2_xlim)
    axins2.set_xticks([])
    axins2.set_yticks([])
    mark_inset(ax, axins2, loc1=1, loc2=3, fc="none", ec="0.5")

    if savepath is not None:
        plt.savefig(savepath, dpi=dpi, bbox_inches="tight")
        print("Saved figure to:", savepath)

    plt.show()
    plt.close(fig)


df_nc = pd.read_csv(NONCRASH_CSV)
df_c  = pd.read_csv(CRASH_CSV)

ttc_nc  = clean_positive(df_nc[TTC_COL])
lttb_nc = clean_positive(df_nc[LTTB_COL])
ttc_c   = clean_positive(df_c[TTC_COL])
lttb_c  = clean_positive(df_c[LTTB_COL])

# Thresholds from NON-CRASH
u_ttc  = np.percentile(ttc_nc, THRESHOLD_Q)
u_lttb = np.percentile(lttb_nc, THRESHOLD_Q)

params_ttc_nc  = fit_gpd(lower_tail_exceedances(ttc_nc, u_ttc))
params_ttc_c   = fit_gpd(lower_tail_exceedances(ttc_c,  u_ttc))

params_lttb_nc = fit_gpd(lower_tail_exceedances(lttb_nc, u_lttb))
params_lttb_c  = fit_gpd(lower_tail_exceedances(lttb_c,  u_lttb))

z_ttc_nc  = 1.0 / np.clip(ttc_nc,  1e-6, None)
z_ttc_c   = 1.0 / np.clip(ttc_c,   1e-6, None)

z_lttb_nc = 1.0 / np.clip(lttb_nc, 1e-6, None)
z_lttb_c  = 1.0 / np.clip(lttb_c,  1e-6, None)

chen_style_density_plot(
    z_ttc_nc, z_ttc_c, u_ttc, params_ttc_nc, params_ttc_c,
    title="Examples of probability density plot of 1/TTC under crash and non-crash conditions",
    xlabel="1/TTC (s$^{-1}$)",
    xlim=XLIM,
    bins=BINS,
    savepath=OUT_TTC_FIG,
    dpi=DPI
)

chen_style_density_plot(
    z_lttb_nc, z_lttb_c, u_lttb, params_lttb_nc, params_lttb_c,
    title="Examples of probability density plot of 1/LTTB under crash and non-crash conditions",
    xlabel="1/LTTB (s$^{-1}$)",
    xlim=XLIM,
    bins=BINS,
    savepath=OUT_LTTB_FIG,
    dpi=DPI
)

print("Thresholds (from non-crash): u_TTC =", u_ttc, ", u_LTTB =", u_lttb)
print("TTC params (non-crash, crash):", params_ttc_nc, params_ttc_c)
print("LTTB params (non-crash, crash):", params_lttb_nc, params_lttb_c)


TypeError: chen_style_density_plot() missing 1 required positional argument: 'title'